In [ ]:
!pip install antropy

import numpy as np
import scipy.io as sio
from tqdm import tqdm
import time
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, skew, kurtosis
from scipy.signal import butter, filtfilt, hilbert, savgol_filter
from sklearn.linear_model import RidgeCV
from scipy.interpolate import Akima1DInterpolator
import matplotlib.pyplot as plt
from antropy import spectral_entropy


In [ ]:
# Preprocessing
from scipy.signal import butter, filtfilt, iirnotch

def apply_car(eeg):
    """Common average referencing"""
    return eeg - np.mean(eeg, axis=1, keepdims=True)

def filter_data(raw_eeg, fs=1000, lowcut=0.1, highcut=200, order=4):
    raw_eeg = apply_car(raw_eeg.astype(np.float64))

    # Bandpass filter
    nyq = 0.5 * fs
    b_band, a_band = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    eeg_bandpassed = filtfilt(b_band, a_band, raw_eeg, axis=0)

    # 60 Hz Notch filter
    notch_freq = 60.0
    Q = 40.0
    b_notch, a_notch = iirnotch(w0=notch_freq / nyq, Q=Q)
    eeg_filtered = filtfilt(b_notch, a_notch, eeg_bandpassed, axis=0)

    return eeg_filtered

def lmp(signal, window_len=100):
    """Local motor potential"""
    return np.convolve(signal, np.ones(window_len)/window_len, mode='same')

def hilbert_bandpower(signal, fs, low, high):
    nyq = 0.5 * fs
    b, a = butter(4, [low/nyq, high/nyq], btype='band')
    band = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(band))
    return np.mean(envelope)


In [ ]:
def get_features(window, fs=1000):
    # Focused feature set: 6 time-domain + 3 high-gamma bands
    bands = [(75, 115), (125, 159), (159, 175)]
    num_channels = window.shape[1]
    features = np.zeros((num_channels, 6 + len(bands)))  # 6 time + 3 freq

    for ch in range(num_channels):
        signal_ch = window[:, ch]
        features[ch, 0] = np.mean(signal_ch)                    # Mean
        features[ch, 1] = np.std(signal_ch)                     # Std
        #features[ch, 2] = kurtosis(signal_ch)
        features[ch, 2] = np.mean(np.diff(signal_ch))           # Derivative
        features[ch, 3] = spectral_entropy(signal_ch, sf=fs, normalize=True)  # Entropy
        features[ch, 4] = np.mean(lmp(signal_ch, window_len=100))             # LMP

        for i, (low, high) in enumerate(bands):
            features[ch, 5 + i] = hilbert_bandpower(signal_ch, fs, low, high)

    return features

def get_windowed_feats(raw_ecog, fs, window_length, window_overlap, lags=[0, 4]):
    """
    Extracts features for each window, including time-lagged windows.
    lags: List of relative time steps to include (e.g., [0, 1] includes t and t-1)
    """
    window_length = int(window_length * fs)
    window_overlap = int(window_overlap * fs)
    step_size = window_length - window_overlap
    clean_data = filter_data(raw_ecog, fs)

    all_feats = []

    # Precompute features per window
    raw_feats = []
    for start in range(0, clean_data.shape[0] - window_length + 1, step_size):
        window = clean_data[start:start + window_length, :]
        feats = get_features(window, fs)
        raw_feats.append(feats.flatten())

    raw_feats = np.array(raw_feats)

    # Build feature matrix with time lags
    for i in range(max(lags), len(raw_feats)):
        lagged = [raw_feats[i - l] for l in lags]
        all_feats.append(np.concatenate(lagged))

    return np.array(all_feats)

In [ ]:
def downsample(flexion, num_windows, shift_ms=37, fs=1000, window_len_ms=100, window_overlap_ms=50):
    """Downsample flexion with a -37 ms causal shift"""
    shift_samples = int((shift_ms / 1000.0) * fs)
    flexion = np.roll(flexion, -shift_samples, axis=0)
    effective_fs = fs / (window_len_ms - window_overlap_ms)
    downsample_factor = int(fs / effective_fs)
    downsampled = flexion[::downsample_factor]
    return downsampled[:num_windows]

In [ ]:
def create_R_matrix(features, N_wind):
    M, num_feats = features.shape
    padded_features = np.vstack([np.tile(features[0], (N_wind - 1, 1)), features])
    R = np.ones((M, 1 + N_wind * num_feats))
    for t in range(M):
        R[t, 1:] = padded_features[t:t + N_wind].flatten()
    return R

In [ ]:
from scipy.signal import savgol_filter

def upsample_predictions(y_pred_downsampled, original_length, window_length, window_overlap, fs=1000, smooth=True):
    y_pred_upsampled = []
    step_size = int((window_length - window_overlap) * fs)
    full_time = np.arange(original_length)

    for subject_data in y_pred_downsampled:
        num_windows = subject_data.shape[0]
        time_pred = np.arange(0, num_windows * step_size, step_size)
        if time_pred[-1] < original_length - 1:
            time_pred = np.append(time_pred, original_length - 1)
            subject_data = np.vstack([subject_data, subject_data[-1]])

        upsampled_data = np.vstack([
            Akima1DInterpolator(time_pred, subject_data[:, i])(full_time)
            for i in range(subject_data.shape[1])
        ]).T

        if smooth:
            for i in range(upsampled_data.shape[1]):
                signal = upsampled_data[:, i]

                # Adaptive smoothing
                local_std = np.std(signal)
                if local_std < 0.02:
                    signal = savgol_filter(signal, window_length=51, polyorder=2, mode='interp')
                else:
                    signal = savgol_filter(signal, window_length=11, polyorder=2, mode='interp')

                upsampled_data[:, i] = signal

        y_pred_upsampled.append(upsampled_data)

    return y_pred_upsampled


In [ ]:
# Load training data

proj_data = sio.loadmat('raw_training_data.mat')
data_glove = proj_data['train_dg'].flatten()
ecog = proj_data['train_ecog'].flatten()

In [ ]:
# Train models

from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from tqdm import tqdm

ensemble_models = []
correlations = []

fs = 1000
window_length = 0.1
window_overlap = 0.05
N_wind = 3

for subj_idx in range(3):
    print(f"\n Subject {subj_idx + 1} Training & Evaluation")

    # Feature extraction
    feats = get_windowed_feats(ecog[subj_idx], fs, window_length, window_overlap)
    R = create_R_matrix(feats, N_wind)
    glove = downsample(data_glove[subj_idx], num_windows=R.shape[0])

    # Split
    split_idx = int(0.95 * len(R))
    X_train, X_val = R[:split_idx], R[split_idx:]
    y_train, y_val = glove[:split_idx], glove[split_idx:]

    subject_models = []
    subject_corr = []

    for f in range(5):
        print(f"Finger {f + 1}")

        # Define base models
        xgb1 = XGBRegressor(n_estimators=400,
                            max_depth=4,
                            learning_rate=0.03,
                            subsample=0.85,
                            colsample_bytree=0.9,
                            tree_method='gpu_hist',
                            predictor='gpu_predictor',
                            device='cuda')
        xgb2 = XGBRegressor(n_estimators=400,
                            max_depth=5,
                            learning_rate=0.02,
                            subsample=0.8,
                            colsample_bytree=0.85,
                            tree_method='gpu_hist',
                            predictor='gpu_predictor',
                            device='cuda')
        xgb3 = XGBRegressor(n_estimators=400,
                            max_depth=5,
                            learning_rate=0.02,
                            subsample=0.8,
                            tree_method='gpu_hist',
                            predictor='gpu_predictor',
                            device='cuda')

        # Train base models
        xgb1.fit(X_train, y_train[:, f], eval_set=[(X_val, y_val[:, f])], verbose=False)
        xgb2.fit(X_train, y_train[:, f], eval_set=[(X_val, y_val[:, f])], verbose=False)
        xgb3.fit(X_train, y_train[:, f], eval_set=[(X_val, y_val[:, f])], verbose=False)

        # Train Meta Ridge
        val_preds = np.stack([xgb1.predict(X_val), xgb2.predict(X_val), xgb3.predict(X_val)], axis=1)
        ridge_meta = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], store_cv_values=True)
        ridge_meta.fit(val_preds, y_val[:, f])

        # Final prediction
        final_preds = ridge_meta.predict(val_preds)
        r, _ = pearsonr(y_val[:, f], final_preds)
        subject_corr.append(r)
        print(f"Correlation: {r:.4f}")

        # Plot
        plt.figure(figsize=(12, 4))
        plt.plot(y_val[:, f], label="Ground Truth", linewidth=2)
        plt.plot(final_preds, label="Prediction", alpha=0.7)
        plt.title(f"Subject {subj_idx + 1} – Finger {f + 1} | r = {r:.3f}")
        plt.xlabel("Time (Windowed)")
        plt.ylabel("Flexion")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        subject_models.append({
            "base_models": (xgb1, xgb2, xgb3),
            "meta_model": ridge_meta
        })

    avg_corr = np.mean([r for i, r in enumerate(subject_corr) if i != 3])
    print(f"Average correlation (excluding Finger 4): {avg_corr:.4f}")
    correlations.append(subject_corr)
    ensemble_models.append(subject_models)

# Final avg
total_avg = sum([r for sub in correlations for i, r in enumerate(sub) if i != 3]) / 12
print(f"\n Overall Average Correlation (excluding Finger 4): {total_avg:.4f}")


In [ ]:
# Predict on Leaderboard data
leaderboard_data = sio.loadmat('leaderboard_data.mat')
predicted_dg_test = []

for i in range(3):
    ecog_test = leaderboard_data['leaderboard_ecog'][i][0]
    feats_test = get_windowed_feats(ecog_test, fs, window_length, window_overlap)
    R_test = create_R_matrix(feats_test, N_wind)
    models = ensemble_models[i]
    Y_pred_downsampled = np.zeros((R_test.shape[0], 5))
    for j in range(5):
        base_preds = np.stack([m.predict(R_test) for m in models[j]["base_models"]], axis=1)
        final_pred = models[j]["meta_model"].predict(base_preds)
        Y_pred_downsampled[:, j] = final_pred
    Y_pred = upsample_predictions([Y_pred_downsampled], ecog_test.shape[0], window_length, window_overlap)[0]
    predicted_dg_test.append(Y_pred)

#predicted_dg = np.array(predicted_dg_test)
#sio.savemat('final_predictions_optimized.mat', {'predicted_dg_raw': predicted_dg})
#print("Saved leaderboard predictions to 'final_predictions_optimized.mat'")

In [ ]:
# Save final predictions
matlab_cell = np.zeros((3, 1), dtype=object)
for i in range(3):
    matlab_cell[i, 0] = predicted_dg_test[i]

sio.savemat('final_predictions.mat', {'predicted_dg': matlab_cell},
          long_field_names=True, do_compression=True)

In [ ]:
print(np.min(matlab_cell[0,0]), np.max(matlab_cell[0,0]))
print(np.min(matlab_cell[1,0]), np.max(matlab_cell[1,0]))
print(np.min(matlab_cell[2,0]), np.max(matlab_cell[2,0]))

-0.7002912144893302 6.474087084304487
-1.2012700646818257 4.904471397146311
-1.0232701089771288 4.46262981253276
